# 11 — Sobol S₁ / Sₜ sensitivity

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ai-systems-today/cubespec/blob/main/notebooks/11_sobol.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/ai-systems-today/cubespec/main?urlpath=lab/tree/notebooks/11_sobol.ipynb) [![nbviewer](https://img.shields.io/badge/render-nbviewer-orange)](https://nbviewer.org/github/ai-systems-today/cubespec/blob/main/notebooks/11_sobol.ipynb) [![Dashboard](https://img.shields.io/badge/open-dashboard-2563eb)](https://sensitive-spark.lovable.app)

**Abstract.** Compute Saltelli's A/B Sobol indices on all three outputs, show convergence vs sample size, and explain the correlation caveat.

**Mirrors.** **Sobol** tab → *N base*, *Seed*, *Output* selector and the warning when correlation is enabled.


In [ ]:
# cubespec-bootstrap
# Install cubespec on first run (Colab/Binder safe; no-op if already importable).
try:
    import cubespec  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'git+https://github.com/ai-systems-today/cubespec.git'])
    import cubespec  # noqa: F401


## 1. S₁ and Sₜ

* `S₁(i)` — variance of `Y` explained by `X_i` alone.
* `Sₜ(i)` — variance of `Y` explained by `X_i` and all interactions involving it.

We compute both for all three outputs at increasing sample sizes to show the Saltelli convergence behaviour.


In [ ]:
import os, pandas as pd
from cubespec import DEFAULT_CSP, sobol_indices
FAST_CI = os.environ.get('CUBESPEC_FAST_CI') == '1'
SAMPLES = (128, 256, 512) if FAST_CI else (128, 256, 512, 1024, 2048)
rows = []
for n in SAMPLES:
    df = sobol_indices(DEFAULT_CSP, n_base=n)
    df.insert(0, 'n_base', n)
    rows.append(df)
all_s = pd.concat(rows, ignore_index=True)
all_s.head()


## 2. Convergence vs n_base for the top P9 driver


In [ ]:
import matplotlib.pyplot as plt
p9 = all_s.query("output == 'P9_compressive_strength'")
leader = p9.query(f'n_base == {SAMPLES[-1]}').sort_values('ST', ascending=False).iloc[0].factor
track = p9.query('factor == @leader').sort_values('n_base')
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(track.n_base, track.S1, 'o-', label='S₁', color='#5b8def')
ax.plot(track.n_base, track.ST, 'o-', label='Sₜ', color='#e94f64')
ax.set_xscale('log', base=2)
ax.set_xlabel('n_base (log scale)')
ax.set_ylabel('Sobol index')
ax.set_title(f'Convergence of {leader} on P9 with sample size')
ax.legend(); fig.tight_layout(); plt.show()


## 3. Final ranking at the largest n_base


In [ ]:
final = (p9.query(f'n_base == {SAMPLES[-1]}')
         .sort_values('ST', ascending=False)
         .reset_index(drop=True))
final


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.barh(final.factor[::-1], final.S1[::-1], color='#5b8def', alpha=0.7, label='S₁')
ax.barh(final.factor[::-1], (final.ST - final.S1)[::-1],
        left=final.S1[::-1], color='#e94f64', alpha=0.7, label='Sₜ − S₁ (interactions)')
ax.set_xlabel('Sobol index'); ax.set_title('P9 sensitivity — main vs interaction share')
ax.legend(); fig.tight_layout(); plt.show()


## 4. Why Sobol is disabled when correlation is on

Saltelli's A/B scheme assumes independent inputs. With a non-trivial correlation matrix the orthogonality breaks down and `S_i` no longer decomposes the variance correctly. The dashboard greys the *Run* button accordingly. Use the *generalised Sobol* indices instead (future work).


In [ ]:
# cubespec-reproducibility-footer
import platform, sys, numpy as np, scipy, pandas as pd, cubespec
print('Reproducibility')
print('  cubespec :', cubespec.__version__)
print('  python   :', sys.version.split()[0], 'on', platform.system())
print('  numpy    :', np.__version__)
print('  scipy    :', scipy.__version__)
print('  pandas   :', pd.__version__)
